## Подключение

Ячейка подключения выполняется первой в каждом сеансе. Она подключает Google Drive (личная папка `practice2026` и общие данные группы `kg_data`), клонирует репозиторий группы в `/content/kg-practice-2026`, переключается на свою ветку `student-NN` с работой группы из `main` и регистрирует модули группы. В первых строках студент указывает свой номер, модуль и фамилию латиницей; токен GitHub лежит в секретах Colab под именем `GITHUB_TOKEN` (`ИНСТРУКЦИЯ_git.md`). Сдача этапа — последняя ячейка блокнота: `repo.submit(...)`.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import sys, os, json, time, subprocess

STUDENT = 'student-07'                     # ← свой номер: вариант 7 — student-07
MY_MODULE = 'm07'                          # ← свой модуль: вариант 7 — m07
SURNAME = 'ivanov'                         # ← фамилия латиницей строчными буквами
REPO = 'github.com/<организация>/kg-practice-2026'     # репозиторий группы

KG = '/content/kg-practice-2026'                       # клон репозитория: код, модули, эталоны, сдача
DATA = '/content/drive/MyDrive/kg_data'                # общие данные группы на Drive (только чтение)
WORK = '/content/drive/MyDrive/practice2026'           # личная папка на Drive: свой набор и результаты
TEST = DATA + '/test_set'                              # общий тестовый набор
TEST_FACTS = WORK + '/test_facts'                      # свои прогоны модулей на тестовом наборе
ONTOLOGY = [KG + '/ontology/MocassinPlus_v2.ttl', KG + '/ontology/SciLibReco.ttl']
os.makedirs(WORK, exist_ok=True)

TOKEN = userdata.get('GITHUB_TOKEN')                   # секрет Colab с токеном GitHub
if not os.path.exists(KG):
    r = subprocess.run(['git', 'clone', '-q', f'https://{TOKEN}@{REPO}.git', KG], capture_output=True, text=True)
    print(r.stderr.replace(TOKEN, '***'))
sys.path.insert(0, KG)
!pip install -q -r {KG}/requirements.txt
from mathkg.gitwork import Repo
repo = Repo(KG, STUDENT, MY_MODULE, SURNAME, token=TOKEN)
repo.start()                                           # своя ветка и всё, что уже принято в main
from mathkg import pipeline, registry
pipeline.setup(student_modules=KG + '/modules')
my_tool = next(n for n, s in registry.REGISTRY.items() if s.module == MY_MODULE)
print('свой модуль:', MY_MODULE, my_tool)

# Этап 02. Текст статьи, токены и ошибки base; первые правила своего модуля

Лекция 2 · сдать к встрече 4 · работа студента: 5–6 ч · вычисления в Colab: секунды на регулярные
выражения и сегментацию; подсчёт токенов на 100 статьях — 1–2 мин на CPU.

## Цель

Студент изучает текст статьи после base как материал для своего модуля: пишет и отлаживает
регулярные выражения для объектов своего модуля, разбирается в нормализации Unicode и сегментации
на предложения, измеряет, сколько токенов занимает статья и формула у кодировщика и у генеративной
модели. Отдельная задача этапа — найти ошибки base на своём наборе и сообщить о них руководителю: base
общая, и её ошибки исправляются централизованно. Итог этапа — копия базовой версии своего модуля с
первыми собственными правилами.


Устройство base разобрано по шагам в отдельном блокноте `notebooks/stage02_разбор_base.ipynb`
(исходник — `этапы/Этап_02_разбор_base.md`): строки PDF со шрифтами, порядок колонок, колонтитулы,
символы, выключные формулы, переносы и абзацы, отметки шрифта, предложения, гиперссылки. Студент
проходит его до шага 4 этого задания: ошибки base удобнее искать, когда видно, на каком шаге они
возникают.

## Что прочитать

* SLP3, гл. 2 «Words and Tokens»: регулярные выражения, нормализация, сегментация, BPE.
* Alammar, Grootendorst, гл. 2, разд. «LLM Tokenization».
* Грейнджер, Тернбулл, Ирвин, гл. 2 «Работа с естественным языком».
* Документация модуля `re` Python (группы, именованные группы, флаги `re.I`, `re.M`).

## Вход

* `practice2026/base/` — свой набор после этапа 1.
* `kg_data/test_set/base/` и `practice2026/test_facts/` — тестовый набор и результаты базовых
  версий модулей.
* Базовая версия своего модуля: `mathkg/modules/mNN_*.py`.

## Шаги

### 1. Регулярные выражения на материале своего модуля

Первая ячейка блокнота — ячейка подключения, та же, что на этапе 1 (Drive, репозиторий группы,
`KG`, `WORK`, `TEST`). Затем
студент собирает образцы текста для своего модуля: все места тестового набора, где встречаются
опорные слова его объектов. Список опорных слов дан в варианте; общий приём одинаков.

In [ ]:
import re, collections
import pandas as pd
from mathkg.base import read_doc

test_ids = pd.read_json(TEST + '/base/docs.jsonl', lines=True).query("status == 'ok'").doc_id.tolist()
docs = {i: read_doc(TEST + '/base', i) for i in test_ids}

def kwic(pattern, docs, width=60, limit=30):
    """Конкорданс: каждое совпадение с контекстом слева и справа."""
    rx = re.compile(pattern, re.I)
    out = []
    for doc_id, d in docs.items():
        for m in rx.finditer(d.text):
            left = d.text[max(0, m.start() - width):m.start()].replace('\n', ' ')
            right = d.text[m.end():m.end() + width].replace('\n', ' ')
            out.append((doc_id, left, m.group(0), right))
    return out[:limit]

for row in kwic(r'\b(Теорема|Theorem|Лемма|Lemma)\s*\d', docs):
    print(row)

Для каждого вида своих объектов студент записывает выражение, проверяет его на конкордансе и
отмечает три вида ошибок: пропуск (объект есть, выражение его не нашло), ложное срабатывание и
неверные границы. Полезные приёмы:

* именованные группы `(?P<num>\d+(?:\.\d+)*)` вместо номеров групп;
* якорь `^` с флагом `re.M` для начала абзаца: в `Doc.text` абзацы разделены пустой строкой;
* граница слова `\b` после ключевого слова: без неё «Definition» находится внутри «definitional»;
* формы русских слов: `лемм\w*` покрывает «лемма», «леммы», «лемме», «леммой».

Готовые выражения базовых версий лежат в `mathkg/modules/_util.py` (`header_re`, `NUM`,
`STATEMENT_WORDS`); студент начинает с них.

### 2. Нормализация Unicode

Base уже заменила лигатуры (ﬁ → fi) и математический курсив (𝑥 → x). Студент проверяет, что
осталось в тексте его набора, и понимает, почему base не применяет NFKC ко всему тексту.

In [ ]:
import unicodedata
d = next(iter(docs.values()))
chars = collections.Counter(c for c in d.text if ord(c) > 127)
for c, n in chars.most_common(30):
    print(repr(c), hex(ord(c)), unicodedata.name(c, '?'), n)

s = 'x² + ½ = 𝑥'
print(unicodedata.normalize('NFC', s), '|', unicodedata.normalize('NFKC', s))

NFKC превращает «x²» в «x2» и «½» в «1⁄2»: для математического текста это потеря смысла. В отчёте
студент перечисляет символы, которые встречаются в его объектах и требуют внимания (тире разных
видов, неразрывный пробел, минус «−» вместо дефиса, буква «ё»).

### 3. Сегментация на предложения

Base делит текст на предложения: русский — библиотекой `razdel`, английский — `pysbd`. Студент
проверяет сегментацию на местах, важных для своего модуля: номера «Lemma 2.1.», сокращения
«т. е.», «см.», «et al.», инициалы авторов, формулы внутри предложения.

In [ ]:
def sentences_with(d, pattern, limit=10):
    rx = re.compile(pattern)
    for s in d.sentences:
        t = d.text[s.start:s.end]
        if rx.search(t):
            print(repr(t[:150]))
            limit -= 1
            if limit == 0:
                break

for d in list(docs.values())[:3]:
    sentences_with(d, r'(т\. е\.|et al\.|Lemma \d+\.\d+\.|[А-Я]\. [А-Я]\.)')

Для сравнения студент делит тот же абзац обеими библиотеками и записывает расхождения:

In [ ]:
from razdel import sentenize
import pysbd
seg = pysbd.Segmenter(language='en', clean=False)
p = d.paragraphs[10]; t = d.text[p.start:p.end]
print([s.text for s in sentenize(t)])
print(seg.segment(t))

### 4. Токены

Модуль на этапах 4 и 6 будет работать с кодировщиком `intfloat/multilingual-e5-base` и генеративной
моделью `Qwen/Qwen2.5-1.5B-Instruct`. Студент измеряет, сколько токенов занимают статья, абзац и
формула у обеих моделей.

In [ ]:
from transformers import AutoTokenizer
tok = {name: AutoTokenizer.from_pretrained(name) for name in
       ['intfloat/multilingual-e5-base', 'Qwen/Qwen2.5-1.5B-Instruct']}

def n_tokens(name, text):
    return len(tok[name].encode(text, add_special_tokens=False))

rows = []
for doc_id, d in docs.items():
    for name in tok:
        rows.append({'doc': doc_id, 'model': name.split('/')[1], 'lang': d.lang,
                     'chars': len(d.text), 'tokens': n_tokens(name, d.text)})
df = pd.DataFrame(rows)
df['chars_per_token'] = df.chars / df.tokens
print(df.groupby(['model', 'lang'])[['tokens', 'chars_per_token']].mean().round(1))

In [ ]:
f = next((f for d in docs.values() for f in d.formulas if f.latex), None)
if f is None:
    print('в тестовом наборе пока нет распознанных формул (LaTeX): сравнение только для текста из PDF')
    f = next(f for d in docs.values() for f in d.formulas)
for name in tok:
    print(name, '| PDF:', tok[name].tokenize(f.text)[:20],
          '| LaTeX:', tok[name].tokenize(f.latex)[:20] if f.latex else '—')

Вывод для отчёта: сколько токенов в средней статье (замер на английской статье в 10 страниц: 7,2 тыс. у Qwen и
7,9 тыс. у e5, около 750 на страницу; русский текст у Qwen даёт в 1,8 раза больше токенов на слово), помещается ли статья
в контекст 512 токенов кодировщика и в 32 тыс. токенов Qwen2.5, во сколько раз русский текст
дороже английского, на какие куски режется LaTeX. Этот вывод обосновывает решение этапа 6: языковая
модель получает отдельные фрагменты, целая статья ей не передаётся.

### 5. Ошибки base

Студент просматривает свой набор и тестовый набор глазами base и ищет ошибки, которые мешают его
модулю: колонтитул внутри абзаца, оторванный номер формулы, абзац, разорванный формулой, перепутанный
порядок колонок, пропущенный заголовок, курсив, не отмеченный в `styled`.

In [ ]:
def page_view(d, page):
    """Абзацы и формулы одной страницы в порядке текста."""
    items = [(p.start, 'P', d.text[p.start:p.end][:100]) for p in d.paragraphs if p.page == page]
    items += [(f.start, 'F' + (f.number or ''), f.text[:80]) for f in d.formulas if f.page == page]
    for _, kind, t in sorted(items):
        print(kind.ljust(6), t)

page_view(docs[test_ids[0]], 3)

Каждая найденная ошибка записывается в задачу (issue) репозитория группы с меткой `base`: в заголовке —
коротко, что не так; в тексте — `doc_id`, страница, что видно, чего ожидалось. Перед записью
студент проверяет, нет ли уже такой задачи, и при совпадении добавляет к ней свой пример. Руководитель исправляет base и сообщает о
новой версии; студент base у себя не правит, иначе позиции фрагментов разойдутся с остальными.

### 6. Копия базовой версии и первые правила

Студент копирует базовую версию своего модуля в папку `modules/` своего клона репозитория, вписывает свою фамилию в
`author` и добавляет первые правила из шага 1. Имя инструмента (`name`), `requires` и формат
результата остаются как в базовой версии.

In [ ]:
import shutil, glob
src = glob.glob(f'{KG}/mathkg/modules/{MY_MODULE}_*.py')[0]
dst = f'{KG}/modules/{os.path.basename(src)}'
if not os.path.exists(dst):
    shutil.copy(src, dst)
print(dst)

В копии относительные импорты базовой версии (`from ..model import …`, `from ._util import …`)
заменяются абсолютными: `from mathkg.model import …`, `from mathkg.modules._util import …`.
Файл редактируется в редакторе Colab или ячейкой `%%writefile`. Он лежит в клоне репозитория в
`/content`, который исчезает с концом сеанса, поэтому незаконченную работу студент тоже отправляет
в свою ветку: `repo.submit('Этап 2: черновик модуля')` (запрос на слияние для черновика не
открывают). Проверка, что студенческая версия заменила базовую:

In [ ]:
from mathkg import pipeline, registry
pipeline.setup(student_modules=KG + '/modules')
my_tool = next(n for n, s in registry.REGISTRY.items() if s.module == MY_MODULE)
print(my_tool, registry.REGISTRY[my_tool].author)
pipeline.run_corpus(TEST + '/base', TEST_FACTS, redo=[my_tool])

`redo` пересчитывает свой модуль и все модули, которые от него зависят. Повторный вызов
`pipeline.setup` перечитывает файл модуля после каждой правки.

## Выход

* Конкорданс и выражения для объектов своего модуля в блокноте;
* задачи с меткой `base` в репозитории группы;
* `modules/mNN_*.py` — копия базовой версии с первыми правилами;
* `practice2026/stage02/REPORT.md` (ячейка сдачи копирует его в `students/NN_фамилия/stage02/`).

## Самопроверка

* Каждое новое выражение проверено на конкордансе тестового набора; для каждого записаны пример
  срабатывания и пример ошибки.
* Студенческая версия модуля загружается: `registry.REGISTRY[my_tool].author` — фамилия студента.
* После `run_corpus(..., redo=[my_tool])` число находок модуля не меньше, чем у базовой версии, либо
  в отчёте объяснено, почему меньше (отсеяны ложные).
* В `base_issues.md` нет повторов уже записанных ошибок.

## Что сдать

Сдача — ячейка сдачи в конце блокнота и запрос на слияние «Этап 2: фамилия» в `main`
(`ИНСТРУКЦИЯ_git.md`, раздел 4). В репозиторий попадают файл модуля `modules/mNN_*.py`, блокнот
`students/NN_фамилия/stage02.ipynb` с выводами и папка `students/NN_фамилия/stage02/` с отчётом
`REPORT.md` и небольшими результатами этапа. В отчёте: выражения с примерами, наблюдения о нормализации и сегментации, таблица токенов по моделям и языкам,
список ошибок base, сравнение числа находок базовой версии и своей на тестовом наборе.

## Как результат станет инструментом агента

Регулярные выражения своего модуля станут первой ступенью инструмента, которым агент будет разбирать
статьи. Измерения токенов определяют, какие куски текста агент сможет передать модели целиком.

## Частые ошибки

* Выражение проверено на одной статье и ломается на статьях другого журнала.
* Отсутствует граница слова после ключевого слова, и находятся «definitional», «Theoremata».
* В копии модуля оставлены относительные импорты `from ..model`, и файл не загружается из папки
  `modules/`.
* Изменено имя инструмента: студенческая версия регистрируется рядом с базовой и не заменяет её.
* Ошибка base исправлена в своей копии текста: позиции фрагментов перестают совпадать с `Doc` у
  остальных студентов.

## Вопросы к защите

1. Чем жадный квантификатор отличается от ленивого? Покажите на примере своего выражения.
2. Почему NFKC не применяется ко всему тексту математической статьи?
3. Какие сокращения мешают делить научный текст на предложения и как с ними справляются `razdel` и
   `pysbd`?
4. Почему одна и та же статья занимает разное число токенов у e5 и у Qwen?
5. Что произойдёт, если передать кодировщику абзац длиной 800 токенов?
6. Почему ошибки base исправляет руководитель, а студент только сообщает о них?

**Вариант.** Опорные слова, примеры объектов и известные слабые места базовой версии своего модуля
перечислены в разделе «Этап 2» файла `варианты/Вариант_NN.md`; конкорданс шага 1 строится по ним.

## Сдача этапа

Перед сдачей блокнот сохраняют (Ctrl+S), а отчёт `REPORT.md` кладут в папку этапа на Drive (`practice2026/stageNN/`). Ячейка копирует в свою папку сдачи `students/NN_фамилия/` блокнот и небольшие файлы этапа (отчёт, таблицы, запросы, журналы), отправляет свои файлы в ветку `student-NN` и печатает ссылку на запрос на слияние в `main`; запрос открывают по ссылке с заголовком «Этап N: фамилия».

In [ ]:
STAGE = 2
repo.save_notebook(f'stage{STAGE:02d}_{SURNAME}.ipynb', STAGE)   # имя своей копии блокнота на Drive
repo.collect(f'{WORK}/stage{STAGE:02d}', STAGE)                  # отчёт и небольшие результаты этапа
repo.submit(f'Этап {STAGE}: {SURNAME}')